# Demo 1: Finding What Breaks — Embedding-Guided Failure Triage

**Hook:** You have thousands of model outputs. How do you find the broken ones without watching them all?

**What this demo shows:**
- Extract video frames and embed with CLIP for instant visual triage
- Failure modes cluster in embedding space — lasso a cluster, find a bug category
- Text-based semantic search lets you query by *description*, not metadata

**Duration:** ~5 minutes | **Dataset:** `quickstart-video` frames (~200-300 frames from 10 videos)

## 1. Extract frames from video dataset

We start with 10 video clips and extract ~1 frame per second. This gives us
hundreds of image samples — enough for UMAP to reveal real structure.
Each frame keeps a reference to its source video and timestamp.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
import fiftyone.brain as fob
import numpy as np
import os
from decord import VideoReader, cpu as decord_cpu
from PIL import Image

# Load source videos
video_dataset = foz.load_zoo_dataset("quickstart-video")

# Extract ~1 frame per second from each video
frames_dir = "/tmp/demo1_frames"
os.makedirs(frames_dir, exist_ok=True)

samples = []
for video_sample in video_dataset:
    vr = VideoReader(video_sample.filepath, ctx=decord_cpu(0))
    fps = vr.get_avg_fps()
    step = max(1, round(fps))  # ~1fps
    indices = list(range(0, len(vr), step))

    video_name = os.path.splitext(os.path.basename(video_sample.filepath))[0]
    for idx in indices:
        frame = vr[idx].asnumpy()
        frame_path = os.path.join(frames_dir, f"{video_name}_f{idx:05d}.jpg")
        Image.fromarray(frame).save(frame_path, quality=85)
        samples.append(fo.Sample(
            filepath=frame_path,
            source_video=video_name,
            frame_index=int(idx),
            timestamp=round(float(idx / fps), 2),
        ))

# Create image dataset (idempotent — safe to re-run)
if "demo1-frames" in fo.list_datasets():
    fo.delete_dataset("demo1-frames")
dataset = fo.Dataset("demo1-frames", persistent=True)
dataset.add_samples(samples)

print(f"Extracted {len(dataset)} frames from {len(video_dataset)} videos")
print(dataset)

In [ ]:
# Compute CLIP embeddings — each sample is now an image, so CLIP works natively.
# We embed with CLIP because its joint text-image space enables semantic search (Step 5).
model_name = "clip"
clip_model = foz.load_zoo_model("clip-vit-base32-torch")
print(f"Embedding {len(dataset)} frames with CLIP ViT-B/32...")

embeddings = []
for i, sample in enumerate(dataset):
    img = np.array(Image.open(sample.filepath).convert("RGB"))
    emb = clip_model.embed(img)
    embeddings.append(emb)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(dataset)} frames embedded")

embeddings = np.array(embeddings)
print(f"Embeddings: {embeddings.shape}")

# Build similarity index (idempotent)
sim_key = f"{model_name}_sim"
if sim_key in dataset.list_brain_runs():
    dataset.delete_brain_run(sim_key)

fob.compute_similarity(
    dataset,
    model="clip-vit-base32-torch",
    embeddings=embeddings,
    brain_key=sim_key,
)
print("Similarity index built — text search enabled via CLIP.")

## 2. Assign failure scores from visual clusters

In production, physics scores come from your validation pipeline. Here we
simulate them *using the embedding structure itself* — frames that look similar
get the same failure type. This mirrors reality: similar-looking scenes tend
to break in the same way (e.g., all reflective-surface frames fail on physics).

In [ ]:
from sklearn.cluster import KMeans
from collections import Counter

# Discover natural visual clusters
N_CLUSTERS = 5
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

cluster_sizes = Counter(cluster_labels)
print("Visual clusters found:")
for c in sorted(cluster_sizes):
    print(f"  Cluster {c}: {cluster_sizes[c]} frames")

# Map clusters to failure types — smallest clusters become failures
# (minority failures are realistic: most outputs are fine, a few break)
sorted_clusters = sorted(cluster_sizes.keys(), key=lambda c: cluster_sizes[c])
failure_map = {}
failure_types_list = ["physics_violation", "temporal_drift", "object_hallucination"]
for i, c in enumerate(sorted_clusters):
    if i < 2:
        failure_map[c] = failure_types_list[i]
    else:
        failure_map[c] = "clean"

print("\nCluster → failure type:")
for c in sorted(failure_map):
    print(f"  Cluster {c} ({cluster_sizes[c]} frames) → {failure_map[c]}")

# Assign scores based on cluster membership
np.random.seed(42)
for i, sample in enumerate(dataset):
    ft = failure_map[cluster_labels[i]]
    sample["failure_type"] = ft
    sample["cluster_id"] = int(cluster_labels[i])

    if ft == "clean":
        sample["physics_score"] = float(np.random.uniform(0.80, 0.98))
        sample["drift_horizon"] = int(np.random.randint(35, 50))
    elif ft == "physics_violation":
        sample["physics_score"] = float(np.random.uniform(0.10, 0.30))
        sample["drift_horizon"] = int(np.random.randint(3, 12))
    elif ft == "temporal_drift":
        sample["physics_score"] = float(np.random.uniform(0.30, 0.50))
        sample["drift_horizon"] = int(np.random.randint(10, 22))
    else:  # object_hallucination
        sample["physics_score"] = float(np.random.uniform(0.20, 0.40))
        sample["drift_horizon"] = int(np.random.randint(8, 18))
    sample.save()

print(f"\nFailure distribution: {dataset.count_values('failure_type')}")

## 3. UMAP visualization — the failure map

UMAP projects 512-dimensional CLIP embeddings down to 2D so you can *see*
the structure. Clusters = groups of visually similar frames. Color by
`physics_score` and the failures light up as distinct regions.

In [ ]:
# Compute 2D UMAP projection (idempotent)
viz_key = f"{model_name}_viz"
if viz_key in dataset.list_brain_runs():
    dataset.delete_brain_run(viz_key)

fob.compute_visualization(
    dataset,
    brain_key=viz_key,
    embeddings=embeddings,
    method="umap",
    num_dims=2,
)
print("UMAP visualization computed.")

## 4. Launch the App — interactive triage

**What to do in the App:**
1. Open the **Embeddings panel** (icon in the top toolbar)
2. Select the brain key `clip_viz`
3. **Color by** `physics_score` — low scores (red/dark) will cluster together
4. **Lasso** a red cluster — you've just found a failure category without inspecting a single frame
5. Click individual samples to see the actual image

**The core insight: aggregate metrics say the model is broken. Embeddings show you *how* it's broken.**

In [ ]:
# Launch FiftyOne App (bound to 0.0.0.0 for remote access)
session = fo.launch_app(dataset, port=5151, address="0.0.0.0")
print("App launched — open the FiftyOne URL in your browser")
print("\n>> Open the Embeddings panel and color by 'physics_score'")
print(">> Lasso a cluster of low-scoring samples to isolate a failure mode")

## 5. Semantic search — query by description

Instead of filtering by metadata fields, you can search by *what the video looks like*.
This is powerful when you don't have pre-existing labels for the failure mode you're hunting.

In [ ]:
# Text-based semantic search: find frames matching a description
# CLIP's joint embedding space lets you query by what the frame LOOKS like
query = "a busy intersection with pedestrians and cars"
results = dataset.sort_by_similarity(
    query,
    brain_key=f"{model_name}_sim",
    k=10,
)

print(f"Top {len(results)} results for '{query}':")
for sample in results:
    print(f"  {sample.filepath.split('/')[-1]}  "
          f"physics={sample.physics_score:.2f}  "
          f"type={sample.failure_type}")

# Update the App view to show search results
session.view = results
print("\n>> Try your own queries: session.view = dataset.sort_by_similarity('your query', brain_key='clip_sim', k=10)")

In [ ]:
# Filter to just the failures — this is your review queue
from fiftyone import ViewField as F

failures = dataset.match(F("physics_score") < 0.5)
print(f"\nFailure queue: {len(failures)} samples need review")
print(f"Failure types: {failures.count_values('failure_type')}")

session.view = failures
print("\n>> App now shows only failures. Review them in the grid.")

## Takeaway

**Aggregate metrics say the model is broken. Embeddings show you *how* it's broken.**

With FiftyOne's embedding visualization + semantic search, you can:
- Triage thousands of frames in minutes, not hours
- Discover failure modes you didn't know to look for
- Build targeted review queues without manual labeling

This scales from ~200 frames (this demo) to millions of frames (production).